## Importing libraries

In [1]:
import numpy as np
import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
# import warnings
# warnings.filterwarnings('ignore')

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Setting device use to MPS
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using Apple Silicon GPU")
else:
    DEVICE = torch.device("cpu")
    print("Using device CPU")

Using Apple Silicon GPU


## Data Loading

Using WikiANN dataset

In [3]:
dataset = load_dataset("wikiann", "en")
dataset

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

In [4]:
# Subsetting the dataset and getting the train, test and validation datasets
train_dataset = dataset["train"].shuffle(seed=SEED).select(range(5000))
test_dataset = dataset["test"].shuffle(seed=SEED).select(range(1000))
val_dataset = dataset["validation"].shuffle(seed=SEED).select(range(1000))

In [5]:
print(train_dataset)
print(test_dataset)
print(val_dataset)

Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'spans'],
    num_rows: 5000
})
Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'spans'],
    num_rows: 1000
})
Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'spans'],
    num_rows: 1000
})


In [6]:
label_list = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_list)

print(f"The list of labels are: {label_list}")
print(f"The number of labels are: {num_labels}")

The list of labels are: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
The number of labels are: 7


## Data Preprocessing

In [ ]:
MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [8]:
id2label = {i:label for i, label in enumerate(label_list)}
label2id = {label:i for i, label in enumerate(label_list)}
print(f"id2label: {id2label}")
print(f"label2id: {label2id}")

id2label: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}
label2id: {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6}


In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
    )

model.to(DEVICE)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

### Helper Functions

In [10]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens
            if word_idx is None:
                label_ids.append(-100)

            # First token of word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            # Subword tokens
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [11]:
train_tokenized = train_dataset.map(tokenize_and_align_labels, batched=True)
val_tokenized = val_dataset.map(tokenize_and_align_labels, batched=True)
test_tokenized = test_dataset.map(tokenize_and_align_labels, batched=True)

## Model Setup

In [12]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [13]:
metric = evaluate.load("seqeval")

In [14]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = [
        [id2label[p] for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    true_labels = [
        [id2label[l] for (p,l) in zip(pred,label) if l != -100]
        for pred,label in zip(predictions,labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

## Training

In [15]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

/Users/msvmilind/Desktop/Innomatics_Tasks/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.625002,0.330505,0.675061,0.756366,0.713405,0.898542
2,0.292468,0.298840,0.731162,0.788025,0.758529,0.912404
3,0.225032,0.299222,0.726696,0.788713,0.756436,0.910134


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/msvmilind/Desktop/Innomatics_Tasks/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/msvmilind/Desktop/Innomatics_Tasks/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=939, training_loss=0.3808338868097502, metrics={'train_runtime': 438.7077, 'train_samples_per_second': 34.191, 'train_steps_per_second': 2.14, 'total_flos': 111765457353264.0, 'train_loss': 0.3808338868097502, 'epoch': 3.0})

## Evaluation

In [20]:
# Workaround for transformers notebook callback bug in some versions/environments
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)
trainer.evaluate(eval_dataset=test_tokenized)

{'eval_loss': 0.2998799979686737,
 'eval_model_preparation_time': 0.0021,
 'eval_precision': 0.7534336167429693,
 'eval_recall': 0.8050314465408805,
 'eval_f1': 0.7783783783783784,
 'eval_accuracy': 0.9108154717910816,
 'eval_runtime': 9.512,
 'eval_samples_per_second': 105.13,
 'eval_steps_per_second': 6.623,
 'epoch': 0}

## Inference

In [23]:
sentence = 'John works at Google in California'

tokens = tokenizer(sentence, return_tensors='pt')
tokens = {k:v.to(DEVICE) for k,v in tokens.items()}
outputs = model(**tokens)
predictions = torch.argmax(outputs.logits, dim=2)
token_list = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])
for token, pred in zip(token_list, predictions[0]):
    if token not in ["[CLS]","[SEP]","[PAD]"]:
        print(token, id2label[int(pred)])

john B-PER
works O
at O
google B-ORG
in O
california I-ORG
